# Wasserstein Distortion example (JAX)

This notebook creates an adversarial attack on VGG-16 Wasserstein Distortion: starting
from random noise, it optimizes an image to minimize the distortion to a reference image.
With a large sigma, the distortion only measures local texture statistics, so the result
should look like a texture resynthesis of the reference.

In [ ]:
# Copyright 2025 Wasserstein Distortion authors.
#
# Licensed under the Apache License, Version 2.0 (the "License"); you may not use this
# file except in compliance with the License. You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software distributed under
# the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
# KIND, either express or implied. See the License for the specific language governing
# permissions and limitations under the License.

import jax
import jax.numpy as jnp
import mediapy as media
import optax
from wasserstein_distortion import jax as wd

In [ ]:
def to_array(image):
    """Converts an image array of shape (height, width, 3) to an array (3, height, width)."""
    return jnp.array(image[..., :3], dtype=jnp.float32).transpose((2, 0, 1))


def to_image(x):
    """Converts an array of shape (3, height, width) to an image array."""
    return jnp.clip(x, 0, 1).transpose((1, 2, 0))


im1 = to_array(media.read_image("./example.png") / 255)
media.show_image(to_image(im1), title="reference")

In [ ]:
wd.load_vgg16_model()
im2 = jax.random.normal(jax.random.key(0), im1.shape)
optimizer = optax.adam(learning_rate=0.1)
opt_state = optimizer.init(im2)
# To test the Wasserstein distortion, we construct a manual log2_sigma map
# with globally log2_sigma = 4
log2_sigma = jnp.full(im1.shape[-2:], 4.0)


@jax.jit
def step(im2, opt_state):
    loss, grad = jax.value_and_grad(wd.vgg16_wasserstein_distortion, argnums=1)(
        im1, im2, log2_sigma
    )
    updates, opt_state = optimizer.update(grad, opt_state)
    return optax.apply_updates(im2, updates), opt_state, loss


for i in range(200):
    im2, opt_state, loss = step(im2, opt_state)
    if i % 20 == 0:
        print(f"step {i}: loss = {loss.item()}")

In [ ]:
media.show_images(
    {
        "reference": to_image(im1),
        "optimized": to_image(im2),
        "difference": to_image((im1 - im2) / 2 + 0.5),
    }
)